In [ ]:
!nvidia-smi

Sat Feb  1 00:59:48 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0


## Naive Block Matmul


multiply - accumulation operation

M x N = P (M: 3x2, N: 2x3, P:3x3)


to calculate P[1,1]

row to multiply from M: blockIdx.y * blockDim.y + threadIdx.y

column to multiply from N: blockIdx.x * blockDim.x + threadIdx.x




how are matrices stored in the DRAM: row major or column major


Row Major:  


            row --> MxN --> M[row*width + i]

            column --> MxN --> N[i*width + col]


            and store in output P


            P[row*width + col]




16x16 Thread Blocks

dim3 blockDim(16, 16) means each block is a 2D array of 16×16 = 256 threads
Threads are arranged in a 16×16 square pattern within each block


```
Block Layout (16x16 threads):
┌─────────────────────┐
│t0,0 t0,1 ... t0,15 │
│t1,0 t1,1 ... t1,15 │
│...                  │
│t15,0 t15,1 ...t15,15│
└─────────────────────┘
```


Grid dimension:

```
For our 5×5 output matrix:

Width: (5 + 16 - 1) / 16 = 1 block
Height: (5 + 16 - 1) / 16 = 1 block
So grid is 1×1 blocks
```

Launching more threads (16×16 = 256) than needed (5×5 = 25)

Threads with indices beyond 5×5 will be idle



The reason for using 16×16 blocks even for a small matrix:

1. Better occupancy for larger matrices

2. Standard block size that works well on most GPUs

3. Powers of 2 are efficient for memory access patterns



In [1]:
%%writefile blocked_matmul.cu
#include <stdio.h>
#include <cuda_runtime.h>


__global__ void blockedMatrixMul(float *A, float *B, float *C,
                         int M, int N, int K) {
    // Calculate global thread indices
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    // Check if thread is within matrix bounds
    if (row < M && col < N) {
        float sum = 0.0f;

        // Each thread computes one element of C
        // by accumulating results into sum
        for (int k = 0; k < K; k++) {
            // Row major access:
            // A[row][k] = A[row * K + k]
            // B[k][col] = B[k * N + col]
            sum += A[row * K + k] * B[k * N + col];
        }

        // Store result in C[row][col]
        C[row * N + col] = sum;
    }
}

int main() {
    // Matrix dimensions
    const int M = 5;  // rows of A
    const int K = 3;  // cols of A, rows of B
    const int N = 5;  // cols of B

    // Host matrices
    float *h_A, *h_B, *h_C;

    // Device matrices
    float *d_A, *d_B, *d_C;

    // Allocate host memory
    h_A = (float*)malloc(M * K * sizeof(float));
    h_B = (float*)malloc(K * N * sizeof(float));
    h_C = (float*)malloc(M * N * sizeof(float));

    // Initialize input matrices with sample values
    // Matrix A (5x3)
    float A_data[15] = {
        1.0, 2.0, 3.0,
        4.0, 5.0, 6.0,
        7.0, 8.0, 9.0,
        10.0, 11.0, 12.0,
        13.0, 14.0, 15.0
    };

    // Matrix B (3x5)
    float B_data[15] = {
        1.0, 2.0, 3.0, 4.0, 5.0,
        6.0, 7.0, 8.0, 9.0, 10.0,
        11.0, 12.0, 13.0, 14.0, 15.0
    };

    // Copy data to host arrays
    memcpy(h_A, A_data, M * K * sizeof(float));
    memcpy(h_B, B_data, K * N * sizeof(float));

    // Allocate device memory
    cudaMalloc(&d_A, M * K * sizeof(float));
    cudaMalloc(&d_B, K * N * sizeof(float));
    cudaMalloc(&d_C, M * N * sizeof(float));

    // Copy matrices from host to device
    cudaMemcpy(d_A, h_A, M * K * sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, K * N * sizeof(float), cudaMemcpyHostToDevice);

    // Define block and grid dimensions
    dim3 blockDim(16, 16);  // 16x16 threads per block
    dim3 gridDim(
        (N + blockDim.x - 1) / blockDim.x,
        (M + blockDim.y - 1) / blockDim.y
    );

    // Launch kernel
    blockedMatrixMul<<<gridDim, blockDim>>>(d_A, d_B, d_C, M, N, K);

    // Copy result back to host
    cudaMemcpy(h_C, d_C, M * N * sizeof(float), cudaMemcpyDeviceToHost);

    // Print input matrices
    printf("Matrix A (5x3):\n");
    for (int i = 0; i < M; i++) {
        for (int j = 0; j < K; j++) {
            printf("%.1f ", h_A[i * K + j]);
        }
        printf("\n");
    }

    printf("\nMatrix B (3x5):\n");
    for (int i = 0; i < K; i++) {
        for (int j = 0; j < N; j++) {
            printf("%.1f ", h_B[i * N + j]);
        }
        printf("\n");
    }

    // Print result matrix
    printf("\nResult Matrix C (5x5):\n");
    for (int i = 0; i < M; i++) {
        for (int j = 0; j < N; j++) {
            printf("%.1f ", h_C[i * N + j]);
        }
        printf("\n");
    }

    // Free memory
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);
    free(h_A);
    free(h_B);
    free(h_C);

    return 0;
}


Writing blocked_matmul.cu


In [2]:
!nvcc -arch=compute_70 -code=sm_70 blocked_matmul.cu -o matmul

In [3]:
! ./matmul

Matrix A (5x3):
1.0 2.0 3.0 
4.0 5.0 6.0 
7.0 8.0 9.0 
10.0 11.0 12.0 
13.0 14.0 15.0 

Matrix B (3x5):
1.0 2.0 3.0 4.0 5.0 
6.0 7.0 8.0 9.0 10.0 
11.0 12.0 13.0 14.0 15.0 

Result Matrix C (5x5):
46.0 52.0 58.0 64.0 70.0 
100.0 115.0 130.0 145.0 160.0 
154.0 178.0 202.0 226.0 250.0 
208.0 241.0 274.0 307.0 340.0 
262.0 304.0 346.0 388.0 430.0 
